# Macro vs ETF Sensitivity Sandbox

Test how fixed-income ETFs move against Treasury yield changes and credit spread changes using ETF Terminal data.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from db.connection import get_engine
from stores.market import PriceStore
from stores.macro import MacroStore

In [ ]:
DATA_BACKEND = "local"
APP_ENV = "uat"
START_DATE = "2024-01-01"
TICKERS = ["TLT", "IEF", "LQD", "HYG"]

In [ ]:
engine = get_engine(data_backend=DATA_BACKEND, app_env=APP_ENV)
price_store = PriceStore(engine)
macro_store = MacroStore(engine)

In [ ]:
price_histories = price_store.get_multi_ticker_price_history(TICKERS, start_date=START_DATE)
prices = pd.DataFrame({ticker: frame["adj_close"] for ticker, frame in price_histories.items()}).dropna()
etf_returns = np.log(prices).diff().dropna()
etf_returns.tail()

In [ ]:
series_ids = ["DGS2", "DGS5", "DGS10", "DGS30", "BAMLC0A0CM", "BAMLH0A0HYM2"]
macro = macro_store.get_series_matrix(series_ids, start_date=START_DATE)
macro.tail()

In [ ]:
rate_changes_bps = macro[["DGS2", "DGS5", "DGS10", "DGS30"]].diff() * 100.0
oas_changes_bps = macro[["BAMLC0A0CM", "BAMLH0A0HYM2"]].diff() * 100.0
factors = pd.concat([rate_changes_bps, oas_changes_bps], axis=1).dropna()
factors.tail()

In [ ]:
aligned = etf_returns.join(factors, how="inner").dropna()
aligned.tail()

In [ ]:
def ols_fit(y: pd.Series, X: pd.DataFrame):
    design = np.column_stack([np.ones(len(X)), X.to_numpy(dtype=float)])
    target = y.to_numpy(dtype=float)
    coeffs = np.linalg.lstsq(design, target, rcond=None)[0]
    fitted = design @ coeffs
    residuals = target - fitted
    total = np.sum((target - target.mean()) ** 2)
    r2 = np.nan if total <= 0 else 1.0 - (np.sum(residuals ** 2) / total)
    return coeffs, r2

In [ ]:
factor_columns = ["DGS2", "DGS5", "DGS10", "DGS30", "BAMLC0A0CM", "BAMLH0A0HYM2"]
rows = []

for ticker in TICKERS:
    y = aligned[ticker]
    X = aligned[factor_columns]
    coeffs, r2 = ols_fit(y, X)
    rows.append({
        "ticker": ticker,
        "intercept": coeffs[0],
        "beta_DGS2": coeffs[1],
        "beta_DGS5": coeffs[2],
        "beta_DGS10": coeffs[3],
        "beta_DGS30": coeffs[4],
        "beta_IG_OAS": coeffs[5],
        "beta_HY_OAS": coeffs[6],
        "r2": r2,
    })

betas = pd.DataFrame(rows).set_index("ticker")
betas

In [ ]:
betas[["beta_DGS2", "beta_DGS5", "beta_DGS10", "beta_DGS30"]].plot(kind="bar", figsize=(12, 5), title="Treasury Factor Betas")
plt.axhline(0, color="black", linewidth=1)
plt.show()

In [ ]:
betas[["beta_IG_OAS", "beta_HY_OAS"]].plot(kind="bar", figsize=(12, 5), title="OAS Factor Betas")
plt.axhline(0, color="black", linewidth=1)
plt.show()

In [ ]:
betas[["r2"]].sort_values("r2", ascending=False)